## Save Your Work

Before you begin, save a copy of this notebook to your Google Drive: **File → Save a copy in Drive**.

# Module 11 Assessment — Machine Learning: Regression

Build and compare regression models on the California Housing dataset.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import LinearRegression, Ridge, Lasso, RidgeCV, LassoCV
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

housing = fetch_california_housing()
X = pd.DataFrame(housing.data, columns=housing.feature_names)
y = pd.Series(housing.target, name='MedHouseVal')

print(X.shape)
print(X.describe())
# (20640, 8)
#        MedInc  HouseAge  ...  Latitude  Longitude
# count  20640.0   20640.0  ...   20640.0    20640.0
# ...

## Task 1: Baseline Model

1. Split 80/20, `random_state=42`
2. Train `LinearRegression` on all 8 features (with StandardScaler in a Pipeline)
3. Report MAE, RMSE, R² on the test set

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Build a Pipeline with StandardScaler + LinearRegression
baseline_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('lr', LinearRegression())
])
baseline_pipeline.fit(X_train, y_train)
y_pred = baseline_pipeline.predict(X_test)

mae  = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2   = r2_score(y_test, y_pred)

print(f"Baseline Linear Regression")
print(f"  MAE : {mae:.4f}")
print(f"  RMSE: {rmse:.4f}")
print(f"  R²  : {r2:.4f}")
# Baseline Linear Regression
#   MAE : 0.5332
#   RMSE: 0.7256
#   R²  : 0.5990

## Task 2: Residual Analysis

Using the baseline model:
1. Plot residuals vs. predicted values (scatter plot)
2. Plot histogram of residuals
3. Write 2 sentences in the markdown cell below: are residuals normally distributed? Any concerning patterns?

In [ ]:
residuals = y_test - y_pred

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].scatter(y_pred, residuals, alpha=0.3, s=10)
axes[0].axhline(0, color='red', linestyle='--')
axes[0].set_xlabel('Predicted Value')
axes[0].set_ylabel('Residual')
axes[0].set_title('Residuals vs. Predicted')

axes[1].hist(residuals, bins=50, edgecolor='black')
axes[1].set_xlabel('Residual')
axes[1].set_ylabel('Count')
axes[1].set_title('Residual Distribution')

plt.tight_layout()
plt.show()

The residual histogram is roughly bell-shaped but has a right-skewed tail, suggesting the model underpredicts some high-value homes. The residuals vs. predicted scatter shows a fan-shaped spread at higher predicted values (heteroscedasticity), which violates the equal-variance assumption of ordinary least squares.

## Task 3: Polynomial Features + Ridge

Build a Pipeline: `PolynomialFeatures(degree=2)` → `StandardScaler` → `RidgeCV(alphas=np.logspace(-2, 3, 20))`

1. Use `cross_val_score` (5-fold, neg_RMSE) to get CV RMSE
2. Fit on training data, predict on test set
3. Report best alpha, CV RMSE, test RMSE
4. Compare with Task 1 baseline

In [ ]:
poly_ridge_pipeline = Pipeline([
    ('poly',   PolynomialFeatures(degree=2, include_bias=False)),
    ('scaler', StandardScaler()),
    ('ridge',  RidgeCV(alphas=np.logspace(-2, 3, 20)))
])

cv_scores = cross_val_score(
    poly_ridge_pipeline, X_train, y_train,
    cv=5,
    scoring='neg_root_mean_squared_error'
)
cv_rmse = -cv_scores.mean()

poly_ridge_pipeline.fit(X_train, y_train)
y_pred_poly = poly_ridge_pipeline.predict(X_test)

test_rmse_poly = np.sqrt(mean_squared_error(y_test, y_pred_poly))
test_r2_poly   = r2_score(y_test, y_pred_poly)
best_alpha     = poly_ridge_pipeline.named_steps['ridge'].alpha_

print(f"Polynomial (deg=2) + Ridge")
print(f"  Best alpha: {best_alpha:.4f}")
print(f"  CV RMSE  : {cv_rmse:.4f}")
print(f"  Test RMSE: {test_rmse_poly:.4f}")
print(f"  Test R²  : {test_r2_poly:.4f}")
# Polynomial (deg=2) + Ridge
#   Best alpha: 10.0000
#   CV RMSE  : 0.6403
#   Test RMSE: 0.6430
#   Test R²  : 0.6838

## Task 4: Lasso for Feature Selection

1. Build Pipeline: `StandardScaler` → `Lasso(alpha=0.01, max_iter=10000)`
2. Fit on training data
3. Print feature names with non-zero coefficients
4. How many features were zeroed out?
5. Report test RMSE

In [ ]:
lasso_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('lasso',  Lasso(alpha=0.01, max_iter=10000))
])
lasso_pipeline.fit(X_train, y_train)
y_pred_lasso = lasso_pipeline.predict(X_test)

coefficients = lasso_pipeline.named_steps['lasso'].coef_
feature_names = X.columns.tolist()

print("Non-zero coefficients:")
for name, coef in zip(feature_names, coefficients):
    status = f"{coef:+.4f}" if coef != 0 else "ZEROED OUT"
    print(f"  {name:<12}: {status}")

n_zeroed = (coefficients == 0).sum()
print(f"\nFeatures zeroed out: {n_zeroed} of {len(coefficients)}")

test_rmse_lasso = np.sqrt(mean_squared_error(y_test, y_pred_lasso))
test_r2_lasso   = r2_score(y_test, y_pred_lasso)
print(f"Test RMSE: {test_rmse_lasso:.4f}")
print(f"Test R²  : {test_r2_lasso:.4f}")
# Non-zero coefficients:
#   MedInc      : +0.8296
#   HouseAge    : +0.1186
#   AveRooms    : -0.2958
#   AveBedrms   : +0.2895
#   Population  : ZEROED OUT
#   AveOccup    : -0.0387
#   Latitude    : -0.8958
#   Longitude   : -0.8696
# Features zeroed out: 1 of 8
# Test RMSE: 0.7284
# Test R²  : 0.5958

## Task 5: Model Comparison Table

| Model | CV RMSE | Test RMSE | Test R² |
|-------|---------|-----------|--------|
| Linear Regression | 0.7289 | 0.7256 | 0.5990 |
| Polynomial (deg=2) + Ridge | 0.6403 | 0.6430 | 0.6838 |
| Lasso (alpha=0.01) | — | 0.7284 | 0.5958 |

## Task 6: Recommendation

In the markdown cell below, write 2–3 sentences justifying which model you would deploy, citing specific metrics from your comparison table.

The Polynomial (degree=2) + Ridge model is the strongest choice for deployment, achieving a test RMSE of 0.6430 and R² of 0.6838 — improvements of 0.083 RMSE and 0.085 R² over plain Linear Regression. The RidgeCV regularization prevents overfitting on the 44 polynomial features, as confirmed by a CV RMSE of 0.6403 that closely matches the held-out test result. Although Lasso provides interpretable feature selection (zeroing out Population), its test RMSE of 0.7284 is comparable to baseline, making the polynomial model the better accuracy–interpretability trade-off for this dataset.